# Nixtla + LightGBM: předpověď znamének dalších deseti minut

Spusťte buňky postupně shora dolů. Připravte alespoň **čtyři dostupné dny** minutových CSV. První dva dny slouží k základnímu učení, třetí ke kalibraci a další dny k časovému vyhodnocení.

Porovnává se MLForecast + LightGBM, přímý klasifikátor LightGBM a jejich kombinace 50:50. Hlavním cílem jsou pravděpodobnosti znamének **jednotlivých minut**, Brierovo skóre a kalibrace.

[Návod a metodika](https://github.com/000307/Odchylky/blob/main/NIXTLA_ZNAMENKA.md) · [Souhrn provedeného experimentu](https://github.com/000307/Odchylky/blob/main/VYSLEDKY_NIXTLA.md)

Knihovny běží bez API klíče. CSV zůstanou v tomto prostředí Colabu; do GitHubu se nenahrávají.
Srovnání zahrnuje původní opakování pátého znaménka i samostatnou sigmoidně
kalibrovanou poslední hodnotu. Přehled starších modelů je v `SROVNANI_MODELU.md`.


## 1. Stáhnout úplné skripty a seznam závislostí

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "000307/Odchylky"
REF = "main"
script_names = [
    "predikce_nixtla_znamenka.py",
    "nixtla_sign_core.py",
    "sign_calibration.py",
    "predikce_linearni_regrese.py",
]
for name in ["requirements_nixtla.txt", *script_names]:
    destination = Path(name)
    urlretrieve(
        f"https://raw.githubusercontent.com/{REPOSITORY}/{REF}/{name}",
        destination,
    )
    if destination.suffix == ".py":
        compile(destination.read_text(encoding="utf-8"), str(destination), "exec")
    print(f"Stažen soubor: {destination} ({destination.stat().st_size} bajtů)")
script = Path("predikce_nixtla_znamenka.py")

## 2. Nainstalovat závislosti

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_nixtla.txt"],
    check=True,
)

## 3. Nahrát vstupní CSV

Vyberte všechny CSV pro společné vyhodnocení. Očekávané sloupce: `Timestamp`, `Čas`, `ODCHYLKA ČR [MW]`. Období nesmí zahrnovat změnu letního a zimního času.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = sorted(name for name in uploaded if name.lower().endswith(".csv"))
if not csv_files:
    raise ValueError("Nahrajte vstupní CSV s minutovými odchylkami.")
print(f"Vybráno CSV souborů: {len(csv_files)}")

## 4. Spustit časové vyhodnocení

Pro každý testovací den se modely učí pouze ze dnů před kalibračním dnem. Nejnovější předchozí den je vyhrazen kalibraci; základní modely se potom na tento den nepřeučují. Váha kombinace zůstává 50:50.

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

SEED = 20261102
TRAIN_DAYS = 3
out_dir = Path("predikce_nixtla")
subprocess.run(
    [sys.executable, str(script), *csv_files,
     "--seed", str(SEED), "--train-days", str(TRAIN_DAYS),
     "--out", str(out_dir)],
    check=True,
    env={**os.environ, "MPLBACKEND": "Agg"},
)
print((out_dir / "vysledky.txt").read_text(encoding="utf-8"))

## 5. Zobrazit skóre, kalibraci a rozdělení dat

**Nižší Brierovo skóre, log-loss a ECE jsou lepší.** U kalibrace sledujte také počty případů. ECE závisí na dělení pravděpodobností do intervalů a samo neprokazuje užitečnost modelu. Srovnání proto obsahuje i jednoduchou četnostní předpověď. Kombinace dvou kalibrovaných pravděpodobností nemusí být automaticky kalibrovaná.

In [ ]:
for file_name in [
    "metriky.csv", "metriky_po_horizontech.csv", "metriky_po_dnech.csv",
    "kalibrace.csv", "rozdeleni_dat.csv", "kalibrace_fitu.csv",
]:
    print(f"\n{file_name}")
    display(pd.read_csv(out_dir / file_name))

for file_name in ["porovnani_brier.png", "kalibrace.png"]:
    display(Image(filename=str(out_dir / file_name)))

## 6. Stáhnout všechny výsledky

In [ ]:
from shutil import make_archive

archive = make_archive("vysledky_nixtla_znamenka", "zip", root_dir=out_dir)
files.download(archive)

## 7. Volitelně: předpověď nového intervalu

Nahraďte ukázkové hodnoty skutečnými pěti měřeními v časovém pořadí a změňte `SPUSTIT_PREDPOVED` na `True`. Příklad znamená, že znáte hodnoty `12:00–12:04` a předpovídáte znaménka pro `12:05–12:14`.

Tato část používá minulá CSV k základnímu učení a kalibraci. Nevystačí jen s pěti aktuálními čísly bez historických dat.

In [ ]:
SPUSTIT_PREDPOVED = False
START = "2026-09-29 12:00"
PRVNICH_PET_MW = [10, 20, 30, 40, 50]  # Nahraďte skutečnými hodnotami.

if SPUSTIT_PREDPOVED:
    forecast_dir = Path("nova_predikce_nixtla")
    subprocess.run(
        [sys.executable, str(script), *csv_files,
         "--seed", str(SEED), "--train-days", str(TRAIN_DAYS),
         "--first-five", *map(str, PRVNICH_PET_MW),
         "--start", START, "--out", str(forecast_dir)],
        check=True,
        env={**os.environ, "MPLBACKEND": "Agg"},
    )
    display(pd.read_csv(forecast_dir / "predikce_noveho_intervalu.csv"))
    forecast_archive = make_archive("nova_predikce_nixtla_znamenka", "zip", root_dir=forecast_dir)
    files.download(forecast_archive)

## Meze interpretace

Přesné nulové skutečnosti se vynechávají z binárního učení, kalibrace a skórování; jejich počet se vykazuje. Pravděpodobnost kladného znaménka jednotlivé minuty není pravděpodobností znaménka desetiminutového průměru.

Dosavadní tři testovací dny byly prohlíženy při předchozích experimentech. Jde o retrospektivní srovnání. Pro konečné potvrzení vybraného postupu je potřeba nová historie; podle nynějšího testu neladit další parametry.